## Planning:
- Load data
- Create sample FE - sum of boolean and proportion of boolean and Digit Position
- Create TE in run.cv
- Implement for XGBoost

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

In [3]:
current_dir = Path.cwd()
competition_root = current_dir.parents[1]

In [4]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [5]:
train = pd.read_csv(competition_path/"data/train.csv")
test = pd.read_csv(competition_path/"data/test.csv")
submit = pd.read_csv(competition_path/"data/sample_submission.csv")

In [6]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()
numeric_cols_ex = numeric_cols.copy()
numeric_cols_ex.remove("is_fraud")

In [7]:
bool_cols = train.select_dtypes(include='bool').columns.tolist()

In [8]:
obj_cols = train.select_dtypes(include='object').columns.tolist()

## Feature Engineering - 1st Digit and 2 first Digit - Benford's Law

In [9]:
def extract_first_digit_features(df, numeric_cols,n=1):
    """Extracts the leading digits (e.g., 1st digit or first 2 digits)

    from numeric columns for Benford's Law analysis.

    Parameters:
    - df: pandas DataFrame
    - numeric_cols: list of columns to process
    - n_digits: number of leading digits to extract (default is 1)
    """
    df = df.copy()

    for col in numeric_cols:
        # 1. Convert to string, strip all non-digit characters (decimals, signs)
        str_series = (
            df[col].astype(str).str.replace(r"\D+", "", regex=True).str.lstrip("0")
        )

        # 2. Extract the first n digits (handling empty strings where values were 0)
        feature_name = (
            f"{col}_first_digit"
        )

        # 3. Extract the first n characters, handle blanks as 'Missing' or NaN,
        # and convert the result into a categorical type
        extracted = (
            str_series.str[:n].replace("", np.nan).fillna("Missing")
        )  # Using string 'Missing' to keep categories clean

        # Extract slice and handle cases where string length is less than n_digits
        df[feature_name] = extracted.astype("category")

    return df

In [10]:
def extract_first_two_digits(df, numeric_cols):
  """Extracts the first two leading digits from numeric columns.

  Handles negatives, decimals, and leading zeros (e.g., 0.0034 -> 34).

  Parameters:
  - df: pandas DataFrame
  - numeric_cols: list of column names to process

  Returns:
  - DataFrame with new columns named '{col}_first_two_digits'
  """
  df = df.copy()

  for col in numeric_cols:
    # 1. Convert to string and remove non-digit characters (signs, decimals, exponents)
    # Then strip any leading zeros
    str_series = (
        df[col].astype(str).str.replace(r"[^\d]", "", regex=True).str.lstrip("0")
    )

    # 2. Extract the first 2 characters, handling empty strings (e.g., original value was 0)
    extracted = str_series.str[:2]

    # 3. Format into a clean numeric feature (using -1 for missing/zero values)
    feature_name = f"{col}_first_two_digits"
    df[feature_name] = (
        extracted.replace("", np.nan).fillna("Missing").astype("category")
    )

  return df

In [11]:
numeric_features = ["amount_usd", "account_balance_usd"]

In [12]:
train_featured = extract_first_digit_features(train, numeric_features)

In [13]:
train_featured = extract_first_two_digits(train_featured, numeric_features)

In [14]:
test_featured = extract_first_digit_features(test, numeric_features)

In [15]:
test_featured = extract_first_two_digits(test_featured, numeric_features)

In [16]:
for col in obj_cols:
    train_featured[col] = train_featured[col].astype("category")
    test_featured[col] = test_featured[col].astype("category")

In [17]:
category_cols = train_featured.select_dtypes(include='category').columns.tolist()

## Building TE Target Encoding with Stratified KFOLD

In [18]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

In [19]:
X = train_featured.drop(columns=["is_fraud"]).copy()
y = train_featured[["id","is_fraud"]].copy()

In [20]:
SEED = 2312
N_FOLDS = 5
TARGET = 'is_fraud'

# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y["is_fraud"]))

In [21]:
# FOLDS

In [22]:
# len(FOLDS[1][0])

## Experiment with Fold

In [23]:
tr_idx = FOLDS[0][0]
va_idx = FOLDS[0][1]

In [24]:
X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
y_tr, y_va = y.iloc[tr_idx].copy(), y.iloc[va_idx].copy()

In [25]:
X_tr.head(2)

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,amount_usd_first_digit,account_balance_usd_first_digit,amount_usd_first_two_digits,account_balance_usd_first_two_digits
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,...,False,False,False,False,False,False,6,1,61,12
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,...,False,False,False,False,False,False,2,3,24,32


In [26]:
X_va.head(2)

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,amount_usd_first_digit,account_balance_usd_first_digit,amount_usd_first_two_digits,account_balance_usd_first_two_digits
8,8,15.06,4.88,5,7.85,47,54,682.02,0,18.5,...,False,False,False,False,False,False,1,6,15,68
10,10,37.50,3.84,1,5.66,47,20,1315.70,0,3.3,...,False,False,False,False,False,False,3,1,37,13


In [27]:
y_tr.head(2)

,id,is_fraud
0,0,0.0
1,1,0.0


In [28]:
y_va.head(2)

,id,is_fraud
8,8,0.0
10,10,0.0


In [29]:
te = TargetEncoder(smooth="auto", target_type='binary')

In [30]:
te.fit(X_tr[category_cols],y_tr['is_fraud'])

TargetEncoder(target_type='binary')

In [31]:
X_va[category_cols] = te.transform(X_va[category_cols])

In [32]:
X_va[category_cols].head(2)

,merchant_category,card_type,auth_method,channel,device_type,amount_usd_first_digit,account_balance_usd_first_digit,amount_usd_first_two_digits,account_balance_usd_first_two_digits
8,0.011733,0.016959,0.014856,0.016617,0.015922,0.016918,0.016126,0.015877,0.016782
10,0.013468,0.016959,0.014856,0.016617,0.015922,0.013615,0.016865,0.013631,0.016792


In [33]:
SEED = 2312
N_FOLDS = 5
TARGET = 'is_fraud'

# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y["is_fraud"]))

In [34]:
def run_te(te_cols): # adding input te_cols for categorical variable required for TE
    encoded_dfs=[]
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        # splitting into training and evaluation
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y.iloc[tr_idx].copy(), y.iloc[va_idx].copy()
        
        # creating TE model for training data
        te = TargetEncoder(smooth="auto",target_type='binary')
        
        # fit on tr data
        te.fit(X_tr[te_cols],y_tr['is_fraud'])

        # transform on tr data
        X_tr[te_cols] = te.transform(X_tr[te_cols])

        # transform on va data
        X_va[te_cols] = te.transform(X_va[te_cols])

        # transform on X_te
        encoded_dfs.append(X_va)
        
    X_te = pd.concat(encoded_dfs,ignore_index=True)
        
    return X_te

In [35]:
train_te = run_te(category_cols)

In [36]:
train_te.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699960 entries, 0 to 699959
Data columns (total 29 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   id                                    699960 non-null  int64  
 1   amount_usd                            699960 non-null  float64
 2   hours_since_last_txn                  699960 non-null  float64
 3   txn_count_last_24h                    699960 non-null  int64  
 4   distance_from_home_km                 699960 non-null  float64
 5   card_age_months                       699960 non-null  int64  
 6   customer_age                          699960 non-null  int64  
 7   account_balance_usd                   699960 non-null  float64
 8   cvv_retry_count                       699960 non-null  int64  
 9   velocity_score                        699960 non-null  float64
 10  time_of_day_hour                      699960 non-null  int64  
 11  

In [37]:
train_featured = train_featured.merge(train_te[['id']+category_cols],on='id', how = 'left')

In [38]:
cat_cols_to_drop = train_featured.select_dtypes(include ="category").columns.tolist()

In [39]:
train_featured = train_featured.drop(columns = cat_cols_to_drop)

In [40]:
train_featured.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699960 entries, 0 to 699959
Data columns (total 30 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   id                                      699960 non-null  int64  
 1   amount_usd                              699960 non-null  float64
 2   hours_since_last_txn                    699960 non-null  float64
 3   txn_count_last_24h                      699960 non-null  int64  
 4   distance_from_home_km                   699960 non-null  float64
 5   card_age_months                         699960 non-null  int64  
 6   customer_age                            699960 non-null  int64  
 7   account_balance_usd                     699960 non-null  float64
 8   cvv_retry_count                         699960 non-null  int64  
 9   velocity_score                          699960 non-null  float64
 10  time_of_day_hour                        6999

In [41]:
# Rename columns ending with '_y' to remove the suffix
train_featured = train_featured.rename(columns=lambda col: col[:-2] if col.endswith('_y') else col)

## XGboost + Optuna

In [43]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

In [28]:
from sklearn.preprocessing import StandardScaler

In [29]:
import xgboost as xgb

In [77]:
import optuna

In [42]:
# define objective using XGBoost
# Step 1: Define the Optuna objective with TimeSeriesSplit on x_train
def objective(trial):
    np.random.seed(SEED)
    params = {
        # fix parameter
        "objective":"binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 5000,
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": SEED,
        "colsample_bytree":0.8,
        "subsample":0.8,
        "n_jobs":-1,

        # trial parameter
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2,log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0,log=True),
        "reg_alpha": trial.suggest_float("alpha", 1e-8, 1.0,log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 30),

        # early stopping round
        "early_stopping_rounds": 200,
    }
    
    train_idx = int(0.85 * X.shape[0])

    x_tr, x_val = X.iloc[:train_idx], X.iloc[train_idx:]
    y_tr, y_val = y.iloc[:train_idx], y.iloc[train_idx:]

    model = xgb.XGBClassifier(**params)
    model.fit(x_tr, y_tr,
              eval_set=[(x_val, y_val)],
              verbose =False)

    preds = model.predict_proba(x_val)[:,1]
    # rmse = np.sqrt(root_mean_squared_error(y_val, preds))
    score_auc = roc_auc_score(y_val, preds)

    # Save best iteration
    trial.set_user_attr("best_iteration", model.best_iteration)

    return score_auc